# Train the Luau task model

This notebook builds the dedicated Luau model used by the **Luau** tab:
short beginner tasks (`Task:` prompts) plus the repository's Luau FAQ
(`Question:` prompts).

Runtime: **GPU (T4 is enough)**. The export uses the same architecture as
Integer GPT, so it plugs into the app's integer-only WASM path. The default
run trains a ~5.2M-parameter model on 12,000 generated rows with the
Luau-focused BPE tokenizer (1,721 tokens, so `local`, `print`, and `for` are
one token each).

Files are written to `MyDrive/tiny-gpt-luau` when Drive is mounted; otherwise
they stay under `/content`.


## 1. Check the runtime


In [ ]:
import os, sys, subprocess
import torch

print(sys.version)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU found. Runtime -> Change runtime type -> T4 GPU.")
    print("Training will still run on CPU, just much more slowly.")


## 2. Get the repository

Set `REPO_URL` to a clone URL you can reach. If the repository is private,
upload a zip of it to `/content` instead and unzip it into
`/content/only-gates-ai` manually.


In [ ]:
REPO_URL = "https://github.com/clinuxrulz/only-gates-ai.git"
REPO_DIR = "/content/only-gates-ai"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("working directory:", os.getcwd())


## 3. Generate and execute-check the task data


In [ ]:
# The JSONL files are committed, so this is optional but keeps the run
# reproducible if the generator changed.
!python3 scripts/luau_tasks.py --out data/luau-tasks --seed 1337

# Luau is required for execution checks and evaluation.
LUAU_ZIP = "/content/luau-ubuntu.zip"
LUAU_DIR = "/content/luau-bin"
subprocess.run([
    "curl", "-L", "-o", LUAU_ZIP,
    "https://github.com/luau-lang/luau/releases/download/0.746/luau-ubuntu.zip",
], check=True)
os.makedirs(LUAU_DIR, exist_ok=True)
subprocess.run(["unzip", "-o", LUAU_ZIP, "-d", LUAU_DIR], check=True)
os.chmod(os.path.join(LUAU_DIR, "luau"), 0o755)
LUAU_BIN = os.path.join(LUAU_DIR, "luau")
print(subprocess.run([LUAU_BIN, "--help"], capture_output=True, text=True).stdout[:200])


In [ ]:
!python3 scripts/check_luau_tasks.py \
  --tasks data/luau-tasks/train.jsonl \
  --tasks data/luau-tasks/val.jsonl \
  --tasks data/luau-tasks/test-seen.jsonl \
  --tasks data/luau-tasks/test-paraphrase.jsonl \
  --tasks data/luau-tasks/test-unseen.jsonl \
  --luau-bin "{LUAU_BIN}" \
  --show-failures 3


## 4. Train the Luau tokenizer

The committed tokenizer is already trained on the task corpus. Re-run this
when the dataset changes; the vocabulary size is printed and must be passed to
the trainer as `--vocab`.


In [ ]:
!python3 scripts/train_luau_tokenizer.py \
  --tasks data/luau-tasks \
  --out data/luau-tokenizer/tokenizer.json


## 5. Train

`--max-minutes` stops cleanly before Colab disconnects. Re-run this cell with
`--resume` to continue from `latest.pt`.

Mount Drive first if you want checkpoints to survive a disconnected runtime.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/tiny-gpt-luau"
os.makedirs(OUT_DIR, exist_ok=True)

!python3 scripts/train_tiny_gpt_luau_colab.py \
  --tasks data/luau-tasks \
  --out "{OUT_DIR}" \
  --tokenizer data/luau-tokenizer/tokenizer.json \
  --vocab 1721 --context 256 \
  --dim 256 --layers 6 --heads 8 --mlp 1024 \
  --epochs 25 --batch 32 --lr 3e-4 \
  --dropout 0.1 --faq-repeat 10 \
  --save-every 500 --max-minutes 120 \
  --export-final


## 6. Evaluate with real Luau execution


In [ ]:
!python3 scripts/eval_luau_tasks.py \
  --model-dir "{OUT_DIR}" \
  --tasks data/luau-tasks/test-seen.jsonl \
  --tasks data/luau-tasks/test-paraphrase.jsonl \
  --tasks data/luau-tasks/test-unseen.jsonl \
  --luau-bin "{LUAU_BIN}" \
  --pass-k 1 \
  --out "/content/luau-eval.json" \
  --show-failures 10


## 7. Publish

Copy `checkpoint.json`, `tokenizer.json`, and `weights.bin` into
`apps/only-gates-ai/public/models/tiny-gpt-luau/`, commit them, and rebuild the
app. The Luau tab loads that folder lazily.


In [ ]:
import shutil

target = os.path.join(REPO_DIR, "apps/only-gates-ai/public/models/tiny-gpt-luau")
os.makedirs(target, exist_ok=True)
for name in ("checkpoint.json", "tokenizer.json", "weights.bin"):
    source = os.path.join(OUT_DIR, name)
    if os.path.exists(source):
        shutil.copy2(source, target)
        print("copied", source)
    else:
        print("missing", source)

# Convenience download if you are not using Drive.
shutil.make_archive("/content/tiny-gpt-luau", "zip", OUT_DIR)
from google.colab import files
files.download("/content/tiny-gpt-luau.zip")
